In [1]:
import pdfplumber
import csv
import re

In [2]:
column_headers = ['cip_year', 'project_type', 'source_page', 'department','project_name','project_id','start_year','end_year', 
                      'previous_appropriations', 'project_total']
years = {}
cip_year='2011'

In [5]:
cleaned = []
headers = []
first_page = True

with pdfplumber.open(r"C:\Users\vince\Documents\GitHub\CIPBD\Dallas\PDF\\" + f"{cip_year}.pdf") as pdf:
    source_page = 0
    for pg in pdf.pages:

        pg_text = pg.extract_text() or ''
        pg_table = pg.extract_table()
        
        source_page += 1

        if pg_table and "District" in pg_text and "Service" in pg_text and not "Funds" in pg_text:
            if first_page:
                headers = pg_table[0]
                first_page = False
            for row in pg_table[1:]:
                
                cleaned_row = [cell.replace('\n', ' ').strip() if cell else '' for cell in row]
                if any('grand total' in str(cell).lower() for cell in cleaned_row):
                    continue
                cleaned_row.append(source_page)
                cleaned.append(cleaned_row)

print(cleaned)
print(headers)

[['Propositions', 'FY 2003-04', 'FY 2004-05', 'FY 2005-06', 'FY 2006-07', 'Total Authorized', 9], ['1. Street and Transportation Improvements', '54,310,762', '52,795,836', '52,666,239', '39,187,163', '198,960,000', 9], ['2. Neighborhood and Community Park, Playground and Recreation Facilities', '10,789,957', '16,042,901', '16,977,213', '13,479,929', '57,290,000', 9], ['3. Library Facilities', '9,538,679', '15,507,437', '14,600,514', '15,878,370', '55,525,000', 9], ['4. Flood Protection and Storm Drainage Facilities', '4,325,657', '3,028,685', '2,120,455', '6,960,203', '16,435,000', 9], ['5. Planning and Designing a Performing Arts Theater and Constructing Related Site Improvements in the Downtown Arts District', '450,000', '0', '1,800,930', '9,004,070', '11,255,000', 9], ['6. City Service and Maintenance Facilities', '16,825,000', '0', '0', '0', '16,825,000', 9], ['7. Animal Control Facilities', '11,755,000', '0', '0', '0', '11,755,000', 9], ['8. Land Acquisition for the Development of

In [ ]:
for i in [7, 8, 9]:
    m = re.search(r'(\d{4})-(\d{2})', headers[i])
    if m:
        yr = "20" + m.group(2)
        years[headers[i]] = yr
        column_headers.append(yr)
    else: 
        years[headers[i]] = cip_year
        column_headers.append(cip_year)
print(years)
print(column_headers)

In [ ]:
# add id, start_year, end_year, and cip_year to rows
ids_raw = {}
ids_clean = []

for row in cleaned:
    
    project_id = ''
    start_year = ''
    end_year = ''
    
    raw_id = row[0][-4:]
    if raw_id in ids_raw: # if id already exists, increment subcount by 1
        ids_raw[raw_id] += 1
    else:
        ids_raw[raw_id] = 1 # if not, set subcount to 1

    project_id = f"{raw_id}.{ids_raw[raw_id]}"

    ids_clean.append(project_id)

    year_cells = []
    for i, cell in enumerate(row):
        if i >= 7 and i < 10:
            year_cells.append((years[headers[i]], cell))

    start_year = next((y for y, cell in year_cells if cell != '0'), '')
    end_year   = next((y for y, cell in reversed(year_cells) if cell != '0'), '')

    for cell in row[:-5]:
        cell = re.sub(r'[\s,()]', '', cell or '')
    print(row)
    
    row.append(project_id)
    row.append(start_year)
    row.append(end_year)
    row.append(cip_year) # cip_year
    
    print(row)

In [ ]:
# up to this point, cleaned rows are in arrangement of 
# project, service, funding source, council district, completion date
# budget, previous_appropriations, y1, y2, y3, future costs, projec_total, source_page, 
# project_id, start year, end year, cip_year

# new arrangement:
# cip_year, project_type, source_page, service, project_name
# project_id, start_year, end_year, previous_appropriations
# project_total, y1, y2, y3, ... everything else

final = []

def clean_num(cell):
    return re.sub(r'[\s,()]', '', cell or '')    

for row in cleaned:
    numeric_indices = {8, 9, 10, 11, 12}  # positions in new_row: previous_appropriations, project_total, y1, y2, y3
    new_order = [16, 1, 12, 2, 0, 13, 14, 15, 6, 11, 7, 8, 9, 3, 4, 5, 10]
    new_row = [row[i] for i in new_order][:-4]
    #new_row = [clean_num(cell) if i in numeric_indices else cell 
    #            for i, cell in enumerate(new_row)]
    final.append(new_row)
    print(new_row)

In [ ]:
with open("outputs/2025.csv", "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(column_headers)
    writer.writerows(final)


In [ ]:
def clean_num(cell):
    return cell.replace(",","").replace(" ","")

print(clean_num("7 50,000"))

In [ ]:
import pdfplumber
from pypdf import PdfReader, PdfWriter

# 1. Rotate the PDF page and save it
reader = PdfReader(r"C:\Users\vince\Documents\GitHub\CIPBD\Dallas\PDF\2021.pdf")
writer = PdfWriter()

for page in reader.pages:
    # Rotate 90, 180, or 270 degrees clockwise
    page.rotate(90)  
    writer.add_page(page)

with open("rotated.pdf", "wb") as f:
    writer.write(f)

# 2. Extract data from the rotated file using pdfplumber
with pdfplumber.open("rotated.pdf") as pdf:
    text = pdf.pages[0].extract_text()
    print(text)